# Fase 4 v2 — 02: videos de overlay de homografía (VideoHomography)

Renderiza, por clip, un video donde se dibuja el **template del campo proyectado**
sobre cada frame vía `inv(H)`, usando `VideoHomography` (camino C estable: solve por
frame + EMA + gate de salto + propagación). Sirve para **ver** la calidad de la
homografía frame a frame: cenital debe pegar bien; lateral mostrará dónde falla.

Salida: `outputs/videos/*.mp4` (h264) + zip descargable.

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
import numpy as np
import cv2, decord

REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core import field_landmarks as fl
from src.core import auto_homography as ah

OUT = REPO / 'notebooks/fase_4_homografia/outputs'
VID = OUT / 'videos'
if VID.exists(): shutil.rmtree(VID)
VID.mkdir(parents=True, exist_ok=True)
print('REPO:', REPO)

In [ ]:
# --- overlay del template via inv(H) ---
def cm_to_img(Hinv, pts_cm):
    p = np.asarray(pts_cm, np.float64).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(p, Hinv).reshape(-1, 2)

CIRC = np.stack([fl.CENTER_CIRCLE[0] + fl.CENTER_CIRCLE[2]*np.cos(np.linspace(0,2*np.pi,60)),
                 fl.CENTER_CIRCLE[1] + fl.CENTER_CIRCLE[2]*np.sin(np.linspace(0,2*np.pi,60))], 1)

def draw_overlay(bgr, H, status):
    img = bgr.copy()
    color_status = {'estimated': (0,255,0), 'propagated': (0,200,255), 'rejected': (0,0,255)}
    txt = status if isinstance(status, str) else str(status)
    cv2.putText(img, txt, (20, 44), cv2.FONT_HERSHEY_SIMPLEX, 1.1,
                color_status.get(txt, (255,255,255)), 3, cv2.LINE_AA)
    if H is None:
        return img
    try:
        Hinv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        return img
    rect = [fl.LANDMARK_POINTS[n] for n in ['inner_tl','inner_tr','inner_br','inner_bl']]
    cv2.polylines(img, [cm_to_img(Hinv, rect).astype(np.int32)], True, (0,255,0), 2, cv2.LINE_AA)
    cl = cm_to_img(Hinv, [fl.LANDMARK_POINTS['center_top'], fl.LANDMARK_POINTS['center_bot']]).astype(np.int32)
    cv2.line(img, tuple(cl[0]), tuple(cl[1]), (0,255,255), 2, cv2.LINE_AA)
    cv2.polylines(img, [cm_to_img(Hinv, CIRC).astype(np.int32)], True, (255,128,0), 2, cv2.LINE_AA)
    for name in ['inner_tl','inner_tr','inner_br','inner_bl','center','postY_top','postY_bot','postB_top','postB_bot']:
        q = cm_to_img(Hinv, [fl.LANDMARK_POINTS[name]])[0]
        if np.all(np.isfinite(q)) and -1e4 < q[0] < 1e4 and -1e4 < q[1] < 1e4:
            cv2.circle(img, (int(q[0]), int(q[1])), 5, (255,0,255), -1, cv2.LINE_AA)
    return img

In [ ]:
CLIPS = {
    'cenital_9933': '/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV',
    'lateral_9913': '/workspace/Meta_Glasses/18abril/Cámaras/IMG_9913.MOV',
}
MAX_OUT_FRAMES = 220   # frames de salida por clip
STEP = 3               # toma 1 de cada STEP frames fuente (acelera)
OUT_W = 960            # ancho de salida
FPS = 15

def transcode_h264(src, dst):
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(src),
                    '-vcodec','libx264','-pix_fmt','yuv420p',str(dst)], check=True)

summary = {}
for name, path in CLIPS.items():
    vr = decord.VideoReader(path)
    n = len(vr)
    idxs = list(range(0, n, STEP))[:MAX_OUT_FRAMES]
    vh = ah.VideoHomography()
    counts = {'estimated':0,'propagated':0,'rejected':0,'none':0}
    f0 = vr[idxs[0]].asnumpy(); H0, W0 = f0.shape[:2]
    scale = OUT_W / W0; out_h = int(round(H0*scale))
    tmp = str(VID / f'{name}_tmp.mp4')
    wr = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (OUT_W, out_h))
    for i in idxs:
        bgr = cv2.cvtColor(vr[i].asnumpy(), cv2.COLOR_RGB2BGR)
        H, status = vh.update(bgr)
        counts[status if status in counts else 'none'] = counts.get(status,0)+1
        ov = draw_overlay(bgr, H, status)
        wr.write(cv2.resize(ov, (OUT_W, out_h)))
    wr.release()
    final = str(VID / f'{name}_overlay.mp4')
    transcode_h264(tmp, final); os.remove(tmp)
    summary[name] = {'frames': len(idxs), **counts, 'mb': round(os.path.getsize(final)/1e6,2)}
    print(name, summary[name])
json.dump(summary, open(VID/'summary.json','w'), indent=2)

In [ ]:
zb = str(OUT / 'overlay_videos')
if os.path.exists(zb+'.zip'): os.remove(zb+'.zip')
shutil.make_archive(zb, 'zip', VID)
print('ZIP:', zb+'.zip', round(os.path.getsize(zb+'.zip')/1e6,2),'MB')
print('videos:', [p.name for p in VID.glob('*.mp4')])

## Resumen

Videos overlay (template proyectado vía `VideoHomography`). Status por frame:
`estimated` (verde, H nueva), `propagated` (ámbar, reusa H previa), `rejected` (roja).
Cenital: mayormente estimated y bien pegado. Lateral: más propagated/none → motiva el
solver multi-feature. Próximo: minimap con objetos (tracks fase_2) sobre la mejor H.